In [15]:
use std::fs::File;
use std::io::{self, Read, Write};
use std::path::Path;

{
    // 1. ファイルパスの設定
    let path = Path::new("drive/drive.img");
    let mut file = File::open(path)?;
    let mut buffer = Vec::new();
    file.read_to_end(&mut buffer)?;

    // 2. "FLA" の位置を特定
    let pattern1 = b"FLA";
    let pos1 = buffer.windows(3).position(|window| window == pattern1);

    // 3. "G_" の位置を特定 (FLAの直後以降で探す)
    let pattern2 = b"G_";
    let pos2 = match pos1 {
        Some(start) => buffer[start + 3..]
            .windows(2)
            .position(|window| window == pattern2)
            .map(|relative_pos| start + 3 + relative_pos),
        None => None,
    };

    match (pos1, pos2) {
        (Some(p1), Some(p2)) => {
            // pos1とpos2の差からオフセットを計算
            let offset = p2 - p1;
            println!("Found 'FLA' at: 0x{:X}", p1);
            println!("Found 'G_' at: 0x{:X}", p2);
            println!("Calculated Offset: 0x{:X} bytes", offset);

            let mut flag = String::new();
            for i in 0..7 {
                let target_pos = p1 + (i * offset);

                if target_pos + 3 <= buffer.len() {
                    let chunk = &buffer[target_pos..target_pos + 3];
                    flag.push_str(&String::from_utf8_lossy(chunk));
                }
            }

            // flag.txt に保存
            let mut output_file = File::create("flag.txt")?;
            output_file.write_all(flag.as_bytes())?;
            
            println!("\nSuccess: FLAG has been saved to 'flag.txt'");
        }
        _ => {
            println!("Could not find the necessary patterns to calculate offset.");
        }
    }

}

Found 'FLA' at: 0x55E210
Found 'G_' at: 0x55E238
Calculated Offset: 0x28 bytes

Success: FLAG has been saved to 'flag.txt'


()